# SPY - Visibility into TrustCall JSON Patch

**Trustcall creates and updates JSON schemas. We may need to access and observ the updates made . TrustCall provides Json doc Id everytime the JSON patch is created. 
That is, when TrustCall updates an existing memory.**

### visibility into the specific changes made by Trustcall?
- we saw before that Trustcall has some of its own tools to:
    - Self-correct from validation failures.
    - Update existing documents

Visibility into these tools can be useful for the agent we're going to build.

## Goal: Adding observability to TrustCall memory creation
- Adding a soy or a listner to Trustcall to gain visibility into its work
- spy documents updates as well as new creation along with its thinking

#### This is especially useful for:
- Debugging complex schemas
- Understanding why a field was updated
- Providing observability for building responsible agents.
- Building agents that react to specific kinds of changes (e.g. “profile updated”, “memory added”, “error fixed”).


In [1]:
import sys
# sys.executable


In [2]:
# Load API key
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_USE_V1"] = "true"


In [3]:
# create genai client and llm
from google import genai

client = genai.Client(api_key = os.environ["GOOGLE_API_KEY"])
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [4]:
# create a llm using any of the above models
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI( model= "gemini-2.5-flash" , 
                              temperature = 0.2 )
llm.invoke("What day is this?").content

'Today is **Monday, May 27, 2024**.'

## create Memory Schema


In [5]:
from pydantic import BaseModel , Field
from typing import List

class Memory(BaseModel):
    content: str = Field(description="Collect information about the user. Example: User name is Dan. User likes to plan a trip to Florida")

class memory_Collection(BaseModel):
    memory: List[Memory] =Field(description= "List of memory contents about the user")



### create a spy class to tools observability

In [6]:
# Inspect the tool calls made by Trustcall
class Spy:
    def __init__(self):
        self.called_tools = []

    def __call__(self, run):
        # Collect information about the tool calls made by the extractor.
        q = [run]
        while q:
            r = q.pop()
            if r.child_runs:
                q.extend(r.child_runs)
            if r.run_type == "chat_model":
                self.called_tools.append(
                    r.outputs["generations"][0][0]["message"]["kwargs"]["tool_calls"]
                )

### Add spy to the TrustCall `with_listeners` extension
```
# Create the extractor
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True,
)
# Add the spy as a listener
trustcall_extractor_see_all_tool_calls = trustcall_extractor.with_listeners(on_end=spy)
```
## OR 


### Create the extractor
```
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True,
    Listners = [spy]
)
```

In [28]:
# trustcall with listeners
from trustcall import create_extractor

# Create the extractor
trustcall_extractor = create_extractor(
    llm,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True,
    enable_updates=True,
)
spy = Spy()
# Add the spy as a listener
trustcall_extractor_see_all_tool_calls = trustcall_extractor.with_listeners(on_end=spy)

### Experiment with trustcall

In [29]:
from langchain_core.messages import AIMessage, SystemMessage , HumanMessage

conversation = [HumanMessage(content="Hi, I'm Diya. I wanted to know if there are any local Libraries here in Irving"), 
                AIMessage(content="Nice to meet you.  There should be. Which state do you live in? "),
                HumanMessage(content=" I live in Dallas, TX.  I want to go eat first")]

sys_inst = SystemMessage(content= "You are a memory extracor. extract user information from the following conversation.")
# extract memory from the conversation
memory_extracted = trustcall_extractor.invoke({'messages': [sys_inst] + conversation})


In [30]:
memory_extracted

{'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'Memory', 'arguments': '{"content": "User name is Diya. User lives in Dallas, TX."}'}, '__gemini_function_call_thought_signatures__': {'2c7eb738-221a-4e7b-80dc-6b83d3bb2691': 'CvUCAQw51sexcSec6tSgP0Duw7W46hXfMnB8Gfaq609Als4GPQ0SmCSYP62guhoej9ins1awONjiSNGjTlVz6uTOrXbcDlfptf2/FGg4CqCaTXJ4a3/ePKBxuD+rB+ZNMwnb9vpLnqeP7pn1Bo4zHh25/M5KDZugfNqiKGo/Zcmfxnw51t/QoDn8bmSIclwu+rSUEk6FUWXxPwyJfjqnDFvo9kmxX3x/Bp+6vuu6PPzxrUo9ifNcrWOh+94YQwE8Fpiuld/jZSTVghZer41Klod8jSxTRJBkq6eFXDMjQMwchUpHRoVlhYo7Hr3VPXI2sGHKezxwc8XTDhd0dGUugMYRSXIWL0itlQ2MaeKM76y13rdL8VnFkquLqpLpF1CkFtJpprL5f5IpggSGltvahAkuLlDliqQ16wX0XKQhl73O6I7ky5uuTAaRU4s4jUCN/mgqIJb9d34OKmPNFgQgp2IOJ4iVroP4wa2/4AhLdA1iWfxZjC3wAw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eb2e6-3842-7af2-997d-ee31c689f1f0-0', tool_calls=[{'name': 'Memory', 'args': {'conte

In [31]:
for m in memory_extracted['messages']:
    m.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  Memory (2c7eb738-221a-4e7b-80dc-6b83d3bb2691)
 Call ID: 2c7eb738-221a-4e7b-80dc-6b83d3bb2691
  Args:
    content: User name is Diya. User lives in Dallas, TX.


In [32]:
memory_extracted['responses'][-1].model_dump()

{'content': 'User name is Diya. User lives in Dallas, TX.'}

## continue conversation and create a new as well as update existing memory with listner

In [33]:
# get existing memory from perevious trustcall 'responses'
# existing memory is a list of tuple : (key , tool_name , value)
existing_memory = [(str(i) , 'Memory' , m.model_dump()) for i, m in  enumerate(memory_extracted['responses'])]if memory_extracted['responses'] else None
existing_memory

[('0', 'Memory', {'content': 'User name is Diya. User lives in Dallas, TX.'})]

In [42]:
#  continue converation and extract memory

updated_conversation = [AIMessage(content="Thats great! You can visit jona's Library in Irving. There is a bekery near by called 'Corner shop' where you can try breakfast and snacks.")
                        ,
                        HumanMessage(content="Nice. I'll try 'Corner Shop' bekery and have some coffee with waffles. I will go bike riding later"),
                       ]

sys_info = SystemMessage(content= '''You are a memory manager.

Your job:
1. Update the existing Memory document ONLY with information that belongs to the same topic.
2. Create a new memory document for any new facts.
3. Do NOT merge all facts into the existing memory.
4. Use the Memory tool for updates.
''')

# get listner
New_memory = trustcall_extractor_see_all_tool_calls.invoke({'messages': [sys_info] + updated_conversation , 
                                         'existing': existing_memory })
New_memory

{'messages': [AIMessage(content='', additional_kwargs={'updated_docs': {}}, response_metadata={}, id='7bc75ba6-a8cb-4f11-8d94-c9c7c42b5b85', tool_calls=[{'name': 'Memory', 'args': {'content': "User will try 'Corner Shop' bakery and have coffee with waffles. User will go bike riding later."}, 'id': 'ef0120c5-97ed-4fd8-9d1c-86120f52de48', 'type': 'tool_call'}], invalid_tool_calls=[])],
 'responses': [Memory(content="User will try 'Corner Shop' bakery and have coffee with waffles. User will go bike riding later.")],
 'response_metadata': [{'id': 'ef0120c5-97ed-4fd8-9d1c-86120f52de48'}],
 'attempts': 1}

In [43]:
# Messages contain the tool calls
for m in New_memory["messages"]:
    m.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  Memory (ef0120c5-97ed-4fd8-9d1c-86120f52de48)
 Call ID: ef0120c5-97ed-4fd8-9d1c-86120f52de48
  Args:
    content: User will try 'Corner Shop' bakery and have coffee with waffles. User will go bike riding later.


In [44]:
for m in New_memory['responses']:
    print(m.model_dump())

{'content': "User will try 'Corner Shop' bakery and have coffee with waffles. User will go bike riding later."}


In [45]:
# Inspect the tool calls made by Trustcall
spy.called_tools

[[{'name': 'PatchDoc',
   'args': {'json_doc_id': '0',
    'planned_edits': "The user's current plans and activities should be added to the existing 'content' field of the Memory document. I will append the new information to the existing content.",
    'patches': [{'path': '/content',
      'value': 'User name is Diya. User lives in Dallas, TX. User will try Corner Shop bakery and have coffee with waffles. User will go bike riding later.',
      'op': 'replace'}]},
   'id': '587f5c18-af6f-4667-ab77-76480f8246e7',
   'type': 'tool_call'}],
 [{'name': 'Memory',
   'args': {'content': "User will try 'Corner Shop' bakery and have coffee with waffles. User will go bike riding later."},
   'id': 'ef0120c5-97ed-4fd8-9d1c-86120f52de48',
   'type': 'tool_call'}]]

## extract information form the spy tool calls for documentation

In [46]:
# create a function to extract information from spy.tool_calls

def extract_tool_info(tool_calls_list , schema_name):
    ''' Extract tool information from apy.tool_Calls for observability
        Args:
            tool_calls: List of tool calls from the model
            schema_name: Name of the schema tool (e.g., "Memory", "ToDo", "Profile") '''
    docs= []
    for tool_call in tool_calls_list:
        for call in tool_call:
            if call['name'] == 'PatchDoc':
                doc = f"""Json patch applied:
                - Memory_document id: {call['args']['json_doc_id']}.. updated
                - Planned_content: {call['args']['planned_edits']} .. created
                - Acutal_content: {call['args']['patches'][0].get('value',None)}..applied\n\n"""
                docs.append(doc+"-"*30)
            elif call['name'] == schema_name:
                doc = f"""New {schema_name} created:
                      - New_memory: {call['args']}\n\n"""
                docs.append(doc+"-"*30)

    return docs

Schema_name = "Memory"
spy_info = spy.called_tools
Changes_Observed = extract_tool_info(spy_info , Schema_name)
Changes_Observed                                  

["Json patch applied:\n                - Memory_document id: 0.. updated\n                - Planned_content: The user's current plans and activities should be added to the existing 'content' field of the Memory document. I will append the new information to the existing content. .. created\n                - Acutal_content: User name is Diya. User lives in Dallas, TX. User will try Corner Shop bakery and have coffee with waffles. User will go bike riding later...applied\n\n------------------------------",
 'New Memory created:\n                      - New_memory: {\'content\': "User will try \'Corner Shop\' bakery and have coffee with waffles. User will go bike riding later."}\n\n------------------------------']

In [47]:
for doc in Changes_Observed:
    print(doc)

Json patch applied:
                - Memory_document id: 0.. updated
                - Planned_content: The user's current plans and activities should be added to the existing 'content' field of the Memory document. I will append the new information to the existing content. .. created
                - Acutal_content: User name is Diya. User lives in Dallas, TX. User will try Corner Shop bakery and have coffee with waffles. User will go bike riding later...applied

------------------------------
New Memory created:
                      - New_memory: {'content': "User will try 'Corner Shop' bakery and have coffee with waffles. User will go bike riding later."}

------------------------------


### HTML viewer for TrustCall patches (notebook‑friendly)

Here’s a compact, pretty HTML viewer you can drop into your notebook to inspect PatchDoc and schema tool calls from your Spy class.

In [48]:
from IPython.display import HTML, display
import html

def build_trustcall_html_viewer(spy_info, schema_name="Memory"):
    """
    spy_info: spy.called_tools  (list of tool_calls per run)
    schema_name: name of your main schema tool, e.g. "Memory" or "UserProfile"
    """

    rows = []

    for run_idx, tool_call_list in enumerate(spy_info):
        for call in tool_call_list:
            name = call.get("name", "")
            args = call.get("args", {})

            if name == "PatchDoc":
                json_id = args.get("json_doc_id", "unknown")
                planned = args.get("planned_edits", [])
                value = args['patches'][-1].get("value", None)

                rows.append({
                    "run": run_idx,
                    "type": "PatchDoc",
                    "id": json_id,
                    "planned": planned,
                    "value": value,
                })

            elif name == schema_name:
                rows.append({
                    "run": run_idx,
                    "type": schema_name,
                    "id": "",
                    "planned": "",
                    "value": args,
                })

    # Build HTML
    html_rows = []
    for r in rows:
        planned_str = html.escape(str(r["planned"]))
        value_str = html.escape(str(r["value"]))

        html_rows.append(f"""
        <div class="tc-card">
          <div class="tc-header">
            <span class="tc-type">{r['type']}</span>
            <span class="tc-run">Run #{r['run']}</span>
          </div>
          <div class="tc-body">
            {'<p><b>json_doc_id:</b> ' + html.escape(str(r['id'])) + '</p>' if r['id'] else ''}
            <p><b>Planned edits:</b></p>
            <pre>{planned_str}</pre>
            <p><b>Actual value:</b></p>
            <pre>{value_str}</pre>
          </div>
        </div>
        """)

    html_page = f"""
    <style>
      .tc-container {{
        font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
        max-width: 900px;
        margin: 1.5rem auto;
      }}
      .tc-title {{
        font-size: 1.4rem;
        font-weight: 600;
        margin-bottom: 0.5rem;
      }}
      .tc-subtitle {{
        color: #666;
        margin-bottom: 1rem;
      }}
      .tc-card {{
        border-radius: 8px;
        border: 1px solid #ddd;
        padding: 0.75rem 1rem;
        margin-bottom: 0.75rem;
        background: #fafafa;
      }}
      .tc-header {{
        display: flex;
        justify-content: space-between;
        margin-bottom: 0.5rem;
      }}
      .tc-type {{
        font-weight: 600;
        color: #1f6feb;
      }}
      .tc-run {{
        font-size: 0.85rem;
        color: #888;
      }}
      .tc-body pre {{
        background: #fff;
        border-radius: 4px;
        padding: 0.5rem;
        border: 1px solid #eee;
        font-size: 0.85rem;
        overflow-x: auto;
      }}
      .tc-body p {{
        margin: 0.3rem 0;
      }}
    </style>
    <div class="tc-container">
      <div class="tc-title">TrustCall Patch & Tool Viewer</div>
      <div class="tc-subtitle">
        Showing PatchDoc calls and <code>{html.escape(schema_name)}</code> creations from Spy.
      </div>
      {''.join(html_rows) if html_rows else '<p>No tool calls recorded.</p>'}
    </div>
    """

    display(HTML(html_page))


In [49]:
spy_info = spy.called_tools   # from your Spy listener
build_trustcall_html_viewer(spy_info, schema_name="Memory")
